# Multi-turn RL on MILES using Qwen3-1.7B on CodeContests

Two containers on `swe-net`:
- **`miles_cc`** (`miles_base:v1`) — Megatron trainer + SGLang rollout + TITO session server + router (GPU).
- **`agent_env`** (`agent_base:v1`) — Harbor orchestrator; runs the **mini-swe-agent** in a per-task Docker sandbox.

Everything runs from this notebook: short steps are foreground `docker exec`; long-running services (Harbor, rollout, training) are launched **detached** with `docker exec -d ... > logfile` and watched via **log-tail cells** (re-run a tail cell to refresh). All paths are derived in the **Config** cell from the repo (identity-mounted) and overridable via environment variables; logs and Harbor trials land under `$WORK_DIR` (default `examples/.../qwen3-codecontests/runtime/work`).

## 0. Prerequisites

- Images **`miles_base:v1`** and **`agent_base:v1`** built.
- An **idle GPU** (pinned via `CC_HIP_VISIBLE_DEVICES`, default GPU 7).

In [10]:
import os, subprocess

# Repo root: explicit $REPO_DIR wins; else auto-detect via git (works on host or
# inside the identity-mounted notebook container).
REPO_DIR  = os.environ.get("REPO_DIR") or subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True, cwd=os.getcwd()).strip()
EX        = os.path.join(REPO_DIR, "examples/experimental/qwen3-codecontests")  # the dir you bash into
RUNTIME   = os.path.join(EX, "runtime")                                          # all runtime data lives here
WORK_DIR  = os.environ.get("WORK_DIR",  os.path.join(RUNTIME, "work"))           # logs + Harbor trials (cc_trials/)
TASKS_DIR = os.environ.get("TASKS_DIR", os.path.join(RUNTIME, "harbor_tasks_cc"))# extracted Harbor task dirs
HF_CACHE  = os.environ.get("HF_CACHE",  os.path.join(RUNTIME, "cache/hf"))       # HuggingFace cache (overridable; can be large)
GPU       = os.environ.get("GPU", "7")                                           # CC_HIP_VISIBLE_DEVICES (pin one idle GPU)
WANDB_KEY = os.environ.get("WANDB_KEY", "")                                      # optional; empty => training runs without W&B

# Export so the `%%bash` cells (and `docker exec -e VAR`) inherit them.
os.environ.update(REPO_DIR=REPO_DIR, EX=EX, RUNTIME=RUNTIME, WORK_DIR=WORK_DIR,
                  TASKS_DIR=TASKS_DIR, HF_CACHE=HF_CACHE, GPU=GPU, WANDB_KEY=WANDB_KEY)
for _d in (WORK_DIR, TASKS_DIR, HF_CACHE):
    os.makedirs(_d, exist_ok=True)

for _k in ("REPO_DIR", "EX", "WORK_DIR", "TASKS_DIR", "HF_CACHE", "GPU"):
    print(f"{_k:9}= {os.environ[_k]}")
print(f"WANDB_KEY= {'set' if WANDB_KEY else '(empty -> W&B disabled)'}")

REPO_DIR = /home/rohith/miles_qwen
EX       = /home/rohith/miles_qwen/examples/experimental/qwen3-codecontests
WORK_DIR = /home/rohith/miles_qwen/examples/experimental/qwen3-codecontests/runtime/work
TASKS_DIR= /home/rohith/miles_qwen/examples/experimental/qwen3-codecontests/runtime/harbor_tasks_cc
HF_CACHE = /home/rohith/miles_qwen/examples/experimental/qwen3-codecontests/runtime/cache/hf
GPU      = 7
WANDB_KEY= (empty -> W&B disabled)


## 1. Host setup & clean slate

Remove our containers and any leftover Harbor task sandboxes from a prior run — deleting a container kills **all background processes inside it**, so nothing lingers. Then disable NUMA balancing (AMD MI300X hygiene) and (re)create the shared network.

In [11]:
%%bash
# remove our containers + any leftover Harbor task sandboxes (= kill their bg processes)
docker rm -f miles_swe agent_env 2>/dev/null
docker rm -f $(docker ps -aq --filter "name=code_contests-") 2>/dev/null || true

# host hygiene + shared network (idempotent; sudo line may prompt — run in a terminal if it errors)
# echo 0 | sudo tee /proc/sys/kernel/numa_balancing
docker network create swe-net 2>/dev/null || echo "swe-net exists"

miles_swe
agent_env
swe-net exists


## 2. Launch the two containers: One for Harbor and one for Miles

 `miles_swe` is privileged with `--ipc host` + large `--shm-size` (SGLang). `agent_env` mounts the Docker socket (Harbor spawns sibling task containers) and the **repo is identity-mounted** (same path inside and out) so the `work`/`tasks`/`trials` paths handed to those sibling containers resolve.

In [12]:
%%bash
# Repo is IDENTITY-mounted (host path == container path) so work/tasks/trials
# under it resolve in the sibling task containers Harbor spawns. HF cache is
# mounted separately (it may live outside the repo) and exposed via HF_HOME.
docker run -d --name miles_swe --hostname miles_swe \
  --network swe-net \
  --privileged --ipc host --shm-size 32g --security-opt label=disable \
  -v "$REPO_DIR":"$REPO_DIR" \
  -v "$HF_CACHE":"$HF_CACHE" \
  -e HF_HOME="$HF_CACHE" \
  miles_base:v1 sleep infinity

docker run -d --name agent_env --hostname agent_env \
  --network swe-net \
  -v /var/run/docker.sock:/var/run/docker.sock \
  -v "$REPO_DIR":"$REPO_DIR" \
  agent_base:v1 sleep infinity

docker ps --format '{{.Names}}\t{{.Image}}\t{{.Status}}' | grep -E 'miles_swe|agent_env'

9724b460482fd6c6d8ec1fa085d2ba1411017b5cf8bcb191016596c656e4beb4


65bec142b22b4f4ecfbe8b5705e5fb7ea7b2d78b49bbea1a65d9ab8b4e418a43
agent_env	agent_base:v1	Up Less than a second
miles_swe	miles_base:v1	Up Less than a second


## 3. Install miles in `miles_swe` 

The base image bakes all deps **except miles itself** (it lives in the bind mount). Foreground `exec` — finishes in seconds and returns.

In [13]:
%%bash
docker exec -e REPO_DIR="$REPO_DIR" miles_swe bash -lc 'cd "$REPO_DIR" && pip install -e . --no-deps --no-build-isolation'
docker exec miles_swe bash -lc 'python3 -c "import miles, miles_plugins.mbridge"' && echo "miles import ok"

Obtaining file:///home/rohith/miles_qwen
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for miles (pyproject.toml): started
  Building editable for miles (pyproject.toml): finished with status 'done'
  Created wheel for miles: filename=miles-0.2.1-0.editable-cp310-cp310-manylinux1_x86_64.whl size=7255 sha256=31574fdac5a447d588d28fe4d675d9e73f0eca02c55833dad8fbb293e191d4cf
  Stored in directory: /tmp/pip-ephem-wheel-cache-5ehc7dps/wheels/39/09/88/93cb0dbf86b2ec2852a3c54558845bb46a4807ef4997fcad34
Successfully built miles



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
[aiter] import [module_aiter_enum] under /app/aiter/aiter/jit/module_aiter_enum.so
/opt/venv/lib/python3.10/site-packages/apex-1.11.0-py3.10-linux-x86_64.egg/apex/transformer/functional/fused_rope.py:49: UserWarning: Aiter backend is selected for fused RoPE. This has lower precision. To disable aiter, export USE_ROCM_AITER_ROPE_BACKEND=0
  warnings.warn("Aiter backend is selected for fused RoPE. This has lower precision. To disable aiter, export USE_ROCM_AITER_ROPE_BACKEND=0", UserWarning)


miles import ok


## 4. Start Harbor (background)

Long-running service → launch **detached** (`docker exec -d`) with output to a log, then wait until `/health` responds before using it. (The host can't reach `swe-net`, so the health check runs from inside `miles_swe`.)

In [14]:
%%bash
docker exec -d -e EX="$EX" -e REPO_DIR="$REPO_DIR" -e WORK_DIR="$WORK_DIR" -e TASKS_DIR="$TASKS_DIR" agent_env bash -lc ' \
    export OPENAI_API_KEY=dummy \
    MSWEA_API_KEY=dummy \
    MSWEA_CONFIG_FILE=$EX/harbor/codecontests.yaml \
    HARBOR_EXTRA_DOCKER_COMPOSE=$EX/harbor/swe_net_override.yaml \
    HARBOR_DELETE_CONTAINERS=true \
    HARBOR_TASKS_DIR=$TASKS_DIR \
    HARBOR_TRIALS_DIR=$WORK_DIR/cc_trials; \
    cd $EX && PYTHONPATH=$REPO_DIR python3 harbor/server.py --port 11000 --max-concurrent 8 > $WORK_DIR/harbor.log 2>&1'

# wait up to ~60s for Harbor (checked from inside miles_swe, which shares swe-net)
docker exec miles_swe bash -lc 'for i in $(seq 1 30); do curl -sf http://agent_env:11000/health && { echo " <- harbor up"; exit 0; }; sleep 2; done; echo "harbor NOT up; log below:"; exit 1'
tail -n 15 "$WORK_DIR/harbor.log" 2>/dev/null

{"status":"ok"} <- harbor up
INFO:     Started server process [15]
INFO:     Waiting for application startup.
2026-06-19 19:21:29,556 __main__ INFO Initialized semaphore with max_concurrent=8
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:11000 (Press CTRL+C to quit)
INFO:     10.200.122.3:45054 - "GET /health HTTP/1.1" 200 OK


## 5. Data preparation

The two cells below build the prompts the agent trains on (run *inside* `miles_swe`, writing to the bind-mounted repo):

1. **Download** — pull [`open-thoughts/CodeContests`](https://huggingface.co/datasets/open-thoughts/CodeContests) from the HF Hub (parquet rows = a `path` id + a gzip-tar `task_binary` per problem).
2. **Convert to Harbor tasks** — untar each problem into `$TASKS_DIR/<task_id>/` (`instruction.md`, `environment/Dockerfile`, `tests/`) — the exact layout Harbor serves per trial.
3. **Split by difficulty** — parse the Codeforces `Rating` in each `instruction.md` into `cc_train_{easy,easy_medium,medium,hard,very_hard,unrated}.jsonl`; we train on `cc_train_easy.jsonl` (rating < 1200) first as a curriculum.


In [15]:
%%bash
# OPTIONAL — fresh data prep: delete everything this operation produces so the
# download + extract + split below rebuild from scratch. (Skip this cell to reuse
# existing data; the next cell is guarded and will no-op if these already exist.)
docker exec -e EX="$EX" -e TASKS_DIR="$TASKS_DIR" -e HF_CACHE="$HF_CACHE" miles_swe bash -lc '
  rm -rf "$TASKS_DIR"/code_contests-*                              # extracted Harbor task dirs
  rm -f  "$EX"/data/cc_train_*.jsonl                              # difficulty-split prompt jsonls
  rm -rf "$HF_CACHE"/hub/datasets--open-thoughts--CodeContests*   # HF dataset cache (forces re-download)
  echo "after clean: tasks=$(ls "$TASKS_DIR" 2>/dev/null | wc -l)  jsonls=$(ls "$EX"/data/cc_train_*.jsonl 2>/dev/null | wc -l)"
'

after clean: tasks=0  jsonls=0


In [16]:
%%bash
# one-time, slow: downloads the HF dataset + extracts 9644 tasks, then splits by difficulty.
# guarded so re-running the cell is a no-op.
docker exec -e EX="$EX" -e TASKS_DIR="$TASKS_DIR" miles_swe bash -lc '
  [ -d "$TASKS_DIR"/code_contests-0000 ] || \
    python3 "$EX"/data_prep/extract_codecontests.py --dataset open-thoughts/CodeContests --out "$TASKS_DIR"
  [ -f "$EX"/data/cc_train_easy.jsonl ] || \
    python3 "$EX"/data_prep/split_by_difficulty.py --tasks "$TASKS_DIR" --out-dir "$EX"/data
'

downloading open-thoughts/CodeContests ...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00,  4.77it/s]s: 100%|██████████| 4/4 [00:00<00:00,  4.77it/s]


  -> /home/rohith/miles_qwen/examples/experimental/qwen3-codecontests/runtime/cache/hf/hub/datasets--open-thoughts--CodeContests/snapshots/11f66f5e81d8035f44c3a576ed6772994d1ed90b
extracting tasks.parquet ...
done: 9644 task dirs under /home/rohith/miles_qwen/examples/experimental/qwen3-codecontests/runtime/harbor_tasks_cc
wrote 9644 samples across 6 buckets -> /home/rohith/miles_qwen/examples/experimental/qwen3-codecontests/data
  cc_train_easy.jsonl : 1026
  cc_train_easy_medium.jsonl : 1134
  cc_train_medium.jsonl : 1373
  cc_train_hard.jsonl : 950
  cc_train_very_hard.jsonl : 933
  cc_train_unrated.jsonl : 4228


### 

**What the agent must do.** Each task's `instruction.md` *is* the prompt. The agent (mini-swe-agent) is instructed to **write a Python solution to `/app/solution.py`** that reads from stdin and writes to stdout, test it on the sample, then submit by issuing `echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT`. The grader (`tests/test.sh`) runs `python3 /app/solution.py` against hidden tests with `pytest`; all pass → `reward.txt = 1`, otherwise `0`.

**Task directory** — `$TASKS_DIR/<task_id>/` (what Harbor serves):
```
code_contests-12701/
├── instruction.md          # the prompt (problem + I/O spec + "write /app/solution.py")
├── environment/Dockerfile  # the per-task sandbox image the agent works in
├── task.toml               # task metadata
└── tests/
    ├── test.sh             # grader: runs solution.py, writes reward.txt (1/0)
    ├── test_state.py       # pytest assertions
    └── test_data.json      # hidden test cases
```

**Trial output** — after a rollout (§7) or training (§8) run, each attempt lands in `$WORK_DIR/cc_trials/<task_id>__<id>/`:
```
agent/mini-swe-agent.txt              # FULL turn-by-turn transcript (human-readable)
agent/mini-swe-agent.trajectory.json  # machine-readable messages
verifier/reward.txt                   # 1 (passed) or 0
result.json                           # trial outcome
```

Below is one such **passed (reward = 1)** trial, shown turn by turn — the prompt, the model's actions, and the environment's responses, each colored distinctly.

### Worked example: A passed trial (`code_contests-13226`. Agent gets reward = 1)

Colors: <span style='color:#2563eb'><b>prompt</b></span> &nbsp; <span style='color:#16a34a'><b>LLM</b></span> &nbsp; <span style='color:#d97706'><b>agent / environment</b></span>

<pre style='white-space:pre-wrap;font-family:monospace;font-size:12px;line-height:1.5'><span style='color:#2563eb'>PROMPT  (instruction.md the model receives)
# 1204_A. BowWow and the Timetable   (Codeforces rating 1000, tag: math)
Trains depart at times 4^k for every integer k &gt;= 0 (1, 4, 16, 64, ...).
You arrive at time s, given in BINARY without leading zeros (0 &lt;= s &lt; 2^100).
Output how many trains departed strictly before s (how many k have 4^k &lt; s).
Examples:  100000000 -&gt; 4    101 -&gt; 2    10100 -&gt; 3
Task: write a Python 3 solution to /app/solution.py (read stdin, write stdout).
Grader runs `python3 /app/solution.py` on hidden tests; submit with
`echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT` once correct.</span>

<span style='color:#16a34a'>step 1 · LLM  — writes the program, but with NO redirect (so it only prints):
  echo &#x27;&lt;program&gt;&#x27;
import sys
def main():
    s = int(sys.stdin.read(), 2)
    count = 0; k = 0
    while True:
        if 4 ** k &lt; s:
            count += 1; k += 1
        else:
            break
    print(count)
if __name__ == &quot;__main__&quot;:
    main()</span>

<span style='color:#d97706'>step 1 · agent  -&gt; returncode 0   (printed to stdout; /app/solution.py NOT created)</span>

<span style='color:#16a34a'>step 2 · LLM (tries to prematurely run the solution):
  python3 /app/solution.py</span>

<span style='color:#d97706'>step 2 · agent  -&gt; returncode 2
  python3: can&#x27;t open file &#x27;/app/solution.py&#x27;: [Errno 2] No such file or directory</span>

<span style='color:#16a34a'>step 3 · LLM  — retries the same run:
  python3 /app/solution.py</span>

<span style='color:#d97706'>step 3 · agent  -&gt; returncode 2
  python3: can&#x27;t open file &#x27;/app/solution.py&#x27;: [Errno 2] No such file or directory</span>

<span style='color:#16a34a'>step 4 · LLM  — now WITH a redirect, so the file is written:
  echo &#x27;&lt;same program&gt;&#x27; &gt; /app/solution.py</span>

<span style='color:#d97706'>step 4 · agent  -&gt; returncode 0   (empty output; file written)</span>

<span style='color:#16a34a'>step 5 · LLM:
  python3 /app/solution.py</span>

<span style='color:#d97706'>step 5 · agent  -&gt; returncode 1
  ValueError: invalid literal for int() with base 2: &#x27;&#x27;   (nothing piped to stdin in this bare run)</span>

<span style='color:#16a34a'>step 6 · LLM  — submit:
  echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT</span>

<span style='color:#d97706'>step 6 · agent  -&gt; action was not executed   (mini-swe-agent submit sentinel; episode ends)</span>

result -&gt; reward = 1   (the grader pipes each hidden test into the saved /app/solution.py, which is correct, so pytest passes)</pre>

## 6. Model

Pre-fetch **`Qwen/Qwen3-1.7B`** into the HF cache (`$HF_CACHE`, default `runtime/cache/hf`, exposed to the container via `HF_HOME`) with `snapshot_download`, so the weights are local before training.

**Optional:** the first rollout/training run (next section) calls the launcher **without** `--skip-prepare`, whose `prepare()` step already downloads `Qwen/Qwen3-1.7B` and then converts it to a Megatron `torch_dist` checkpoint. This cell just separates the ~3.4 GB download from that first run — skip it if you don't mind the first run doing the download itself.

In [17]:
%%bash
docker exec miles_swe bash -lc 'python3 -c "from huggingface_hub import snapshot_download; snapshot_download(\"Qwen/Qwen3-1.7B\")"'

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 210592.67it/s]


## 8. Training run 

The real GRPO loop (rollout → reward → update → weight-sync → checkpoint). `--skip-prepare` reuses the checkpoint converted in Section 7 (the rollout). Paste your W&B key, launch detached, watch the tail cell.

**One run at a time:** If am earlier run is still live, reset first (Section 10) — stacking a second ray/SGLang cluster on the GPU causes agent `500`/`404` errors.

In [20]:
%%bash
docker exec -d -e EX="$EX" -e REPO_DIR="$REPO_DIR" -e WORK_DIR="$WORK_DIR" -e TASKS_DIR="$TASKS_DIR" -e GPU="$GPU" -e WANDB_KEY="$WANDB_KEY" miles_swe bash -lc 'cd "$EX"; \
    export CC_HIP_VISIBLE_DEVICES=$GPU \
    WANDB_KEY=$WANDB_KEY \
    MILES_ROUTER_EXTERNAL_HOST=miles_swe \
    AGENT_SERVER_URL=http://agent_env:11000 \
    HARBOR_TASKS_DIR=$TASKS_DIR \
    WANDB_DIR=$WORK_DIR/wandb; \
    PYTHONPATH=$REPO_DIR python3 run-qwen3-codecontests.py --skip-prepare \
    --prompt-data $EX/data/cc_train_easy.jsonl \
    --num-rollout 20 \
    --rollout-batch-size 2 \
    --n-samples-per-prompt 8 \
    --global-batch-size 16 \
    --max-seq-len 16384 --save-interval 2 > $WORK_DIR/train.log 2>&1'
echo "training launched (background); watch the tail cell below"

training launched (background); watch the tail cell below


## 9. Monitoring Training

One cell drives the whole run. Every refresh it prints, **pinned at the top so it never scrolls away**, the **W&B run link** (parsed from `train.log`) followed by a **live `train.log` tail**. Then:

- **During the rollout phase** (before the first training step finishes) it **reuses the Section 7 rollout monitor** (`rollout_panel`) to show the per-sample turns/reward view — so run that cell once first.
- **After the first full rollout + training step**, it switches to the live, **hover-able** line charts below.

Live, **hover-able** line charts rendered **in the notebook** (so a non-W&B viewer sees the same curves), parsed from `train.log`:

- **Timing** — `step_time` (wall-clock between steps) vs `rollout_time` (`perf/rollout_time`).
- **Reward** — `rollout/raw_reward` (pass rate) per step.
- **Truncated** — `rollout/truncated` per step.

Hover any point to read its **Y value** (plotly; auto-installed on first run, `hovermode="x unified"` shows all series at that step). Below the charts: a per-step table (re-printed every refresh, so every step is shown) and the **turns-per-sample** for the latest rollout. Re-renders in place; **interrupt (stop) to end** — this only reads logs and does not affect training. These mirror the W&B `perf/*`, `rollout/*` series.

In [21]:
# Live training monitor: W&B link + train.log tail + current-step rollout panel +
# hover-able charts. Definitions live in notebook-monitoring/training_monitor.py. Interrupt to end.
import os, sys, importlib
sys.path.insert(0, os.path.join(
    os.environ.get("EX", os.path.join(os.getcwd(), "examples/experimental/qwen3-codecontests")),
    "notebook-monitoring"))
import rollout_monitor, training_monitor
importlib.reload(rollout_monitor); importlib.reload(training_monitor)
training_monitor.watch_training()


W&B run: (link not in train.log yet)

=== train.log (tail) ===
(waiting for log...)

=== step 2 rollout: 2 problems | 16 samples | submitted 12 | running 4 | other 0 | reward>0: 3 ===
-- Submitted (12) --
[0] code_contests-3381  reward=0.0  turns=3
     cmd: echo 'import sys def main(): data = sys.stdin.read().split() a = int(data[0]) b = int(data[1]) s = int(data[2]) D = abs(
     out: <returncode>0</returncode> <output> </output>
[1] code_contests-3381  reward=1.0  turns=4
     cmd: python3 /app/solution.py
     out: <returncode>1</returncode> <output> Traceback (most recent call last): File "/app/solution.py", line 18, in <module> mai
[2] code_contests-3381  reward=1.0  turns=4
     cmd: mkdir -p /app && cat <<'EOF' > /app/solution.py import sys def main(): data = sys.stdin.read().split() a = int(data[0]) 
     out: <returncode>0</returncode> <output> </output>
[3] code_contests-1890  reward=0.0  turns=5
     cmd: python3 /app/solution.py
     out: <returncode>0</returncode> <output

step  step_t  roll_t  reward  trunc
   0       -     383   0.250  0.438
   1     521     501   0.250  0.062
step_t = wall-clock between consecutive step completions; the first step has no predecessor, so it shows '-'.

[stopped]


## 10. Stop / reset between runs

**Run only ONE rollout/training at a time.** Launching a second while one is live stacks a second ray cluster + SGLang engine on the GPU, which causes endpoint errors (`500`/`404`/`400`) in the agent. `pkill` alone leaves ray clusters and zombie workers behind, so the clean reset is to **restart the trainer container** — it clears the whole process namespace but keeps the image + the `pip install -e` (writable layer survives restart). Re-run cell 9 (Harbor) only if you also restarted `agent_env`.

In [ ]:
%%bash
# Reset the trainer between runs: restart clears ALL ray/sglang clusters + zombie workers
# (pkill alone leaves them, which stacks runs -> endpoint 500/404 errors). Keeps the image
# and the `pip install -e` (writable layer survives restart). Takes ~20s.
docker restart miles_swe
docker exec miles_swe bash -lc 'rm -rf /tmp/ray/session_* 2>/dev/null; rm -f /app/aiter/aiter/jit/build/lock_* 2>/dev/null'
echo "miles_swe reset; live ray/sglang: $(docker exec miles_swe bash -lc 'pgrep -fc "raylet|sglang|train.py" 2>/dev/null')"

# full teardown (uncomment to remove containers entirely):
# docker rm -f miles_swe agent_env
# docker rm -f $(docker ps -aq --filter "name=code_contests-") 2>/dev/null || true